# LM for Sales — GCS Data Preparation and EDA

**Project:** `my-project-data-handling`  
**Project alias:** LM for Sales  
**Dataset:** 2019 sales transactions (431,709 rows reported in Milestone 1)  
**Task:** Database-grounded natural-language question answering for sales analytics

This notebook adapts the structure of the provided IMDB GCS preparation notebook to the sales project described in Milestone 1.

### Workflow

1. Authenticate with Google Cloud and configure storage.
2. Download the original CSV from Google Cloud Storage (GCS).
3. Inspect the schema and validate the data.
4. Perform EDA: missingness, duplicates, revenue distributions, monthly and country summaries, and product rankings.
5. Create a reproducible **70% train / 15% development / 15% test** split with approximate stratification by month and country (and product category if available).
6. Save processed data as versioned Parquet and write JSON metadata.
7. Optionally load the processed Parquet into BigQuery.
8. Verify outputs and compare selected Python aggregates with BigQuery.

**Important:** Milestone 1 specifies the dataset size and time range, but does not name the original CSV columns, source URL, GCP project ID, bucket, or object path. Configure these below. This notebook does not invent those details. It also does not claim results until the cells have been executed in your environment.

## 1. Import Libraries and Configure GCP

In [ ]:
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from google.colab import auth
from google.cloud import storage, bigquery

# Authenticate the current Google Colab session.
auth.authenticate_user()

# -----------------------------
# REQUIRED GCP CONFIGURATION
# -----------------------------
PROJECT_ID = "my-project-data-handling"
BUCKET_NAME = "sales-dh"
LOCATION = "EU"

# Path to the original CSV object in the bucket.
RAW_GCS_PATH = "raw/sales_transactions_2019.csv"

# Output prefixes
PROCESSED_PREFIX = "processed"
SPLIT_PREFIX = "splits"

# Optional BigQuery destination
BQ_DATASET = "sales_analytics"
BQ_TABLE_PREFIX = "sales_transactions"

# Local working paths
LOCAL_RAW_PATH = "sales_transactions_2019.csv"
LOCAL_OUTPUT_DIR = "sales_project_output"

# Reproducible split seed
RANDOM_STATE = 42

if PROJECT_ID == "YOUR_GCP_PROJECT_ID" or BUCKET_NAME == "YOUR_GCS_BUCKET":
    raise ValueError("Set PROJECT_ID and BUCKET_NAME before running cloud steps.")

print(f"Project: {PROJECT_ID}")
print(f"Bucket: gs://{BUCKET_NAME}")
print(f"Raw object: gs://{BUCKET_NAME}/{RAW_GCS_PATH}")

## 2. Connect to the Existing GCS Bucket

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.lookup_bucket(BUCKET_NAME)

if bucket is None:
    raise ValueError(
        f"Bucket gs://{BUCKET_NAME} was not found or is not accessible. "
        "Check the bucket name, project, permissions, and authentication."
    )

print(f"Connected successfully to gs://{BUCKET_NAME}")

## 3. Download the Raw Sales Dataset from GCS

In [ ]:
raw_blob = storage_client.bucket(BUCKET_NAME).blob(RAW_GCS_PATH)

if not raw_blob.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at gs://{BUCKET_NAME}/{RAW_GCS_PATH}. "
        "Upload the original CSV or update RAW_GCS_PATH."
    )

raw_blob.download_to_filename(LOCAL_RAW_PATH)

print(f"Downloaded gs://{BUCKET_NAME}/{RAW_GCS_PATH}")
print(f"Saved locally as: {LOCAL_RAW_PATH}")
print(f"File size: {os.path.getsize(LOCAL_RAW_PATH):,} bytes")

# Keep the original source unchanged. All transformations are applied to copies.
source_sha256 = hashlib.sha256(Path(LOCAL_RAW_PATH).read_bytes()).hexdigest()
print("Raw file SHA-256:", source_sha256)

## 4. Load and Validate the Sales Dataset

The Milestone 1 document reports 431,709 rows from 2019. The checks below compare the observed file with that description, but do not discard rows automatically. Column names are inspected first because the PDF does not provide a schema.

In [ ]:
df_raw = pd.read_csv(LOCAL_RAW_PATH, low_memory=False)
df = df_raw.copy()

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types:")
display(df.dtypes.rename("dtype").to_frame())
print("\nFirst five rows:")
display(df.head())

EXPECTED_ROWS_FROM_MILESTONE_1 = 431_709
if len(df) != EXPECTED_ROWS_FROM_MILESTONE_1:
    print(
        f"WARNING: observed {len(df):,} rows; Milestone 1 reports "
        f"{EXPECTED_ROWS_FROM_MILESTONE_1:,}. Investigate whether this is "
        "a different source version or whether rows were filtered."
    )
else:
    print("Observed row count matches the Milestone 1 description.")

print("\nExact duplicate rows:", int(df.duplicated().sum()))

## 5. Configure and Detect Analytical Columns

The PDF does not list exact source column names. The notebook tries common names, then asks you to confirm or override the mappings. Do not proceed with revenue analysis until these mappings match the real dataset.

Revenue should use a source revenue/amount field if its meaning is documented. Otherwise, it may be derived from quantity × unit price only when that calculation matches the dataset's business rules (including any discounts, returns, and cancellations).

In [ ]:
def find_column(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None

# Automatically suggested mappings (review these carefully).
DATE_COL = find_column(df.columns, [
    "invoicedate", "invoice_date", "transaction_date", "date", "order_date"
])
COUNTRY_COL = find_column(df.columns, ["country", "country_name", "market"])
PRODUCT_COL = find_column(df.columns, [
    "stockcode", "stock_code", "product_id", "product_code", "sku", "product"
])
PRODUCT_DESC_COL = find_column(df.columns, [
    "description", "product_description", "product_name", "item_description"
])
QUANTITY_COL = find_column(df.columns, ["quantity", "qty", "units"])
UNIT_PRICE_COL = find_column(df.columns, [
    "unitprice", "unit_price", "price", "item_price"
])
REVENUE_COL = find_column(df.columns, [
    "revenue", "sales", "sales_amount", "line_total", "amount", "total"
])

# If automatic detection is incorrect, replace a value with the exact column name.
# Example: DATE_COL = "InvoiceDate"

print({
    "date": DATE_COL,
    "country": COUNTRY_COL,
    "product": PRODUCT_COL,
    "product_description": PRODUCT_DESC_COL,
    "quantity": QUANTITY_COL,
    "unit_price": UNIT_PRICE_COL,
    "revenue_or_amount": REVENUE_COL
})

if DATE_COL is None:
    raise ValueError("Set DATE_COL to the actual transaction date column.")
if COUNTRY_COL is None:
    raise ValueError("Set COUNTRY_COL to the actual country column.")
if PRODUCT_COL is None:
    raise ValueError("Set PRODUCT_COL to the actual product identifier/name column.")

# Parse date values; unparseable values become NaT and are measured below.
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

# Prefer a documented revenue/amount field; otherwise calculate from quantity and price.
if REVENUE_COL is not None:
    df["_analysis_revenue"] = pd.to_numeric(df[REVENUE_COL], errors="coerce")
    REVENUE_BASIS = f"source column: {REVENUE_COL}"
elif QUANTITY_COL is not None and UNIT_PRICE_COL is not None:
    df[QUANTITY_COL] = pd.to_numeric(df[QUANTITY_COL], errors="coerce")
    df[UNIT_PRICE_COL] = pd.to_numeric(df[UNIT_PRICE_COL], errors="coerce")
    df["_analysis_revenue"] = df[QUANTITY_COL] * df[UNIT_PRICE_COL]
    REVENUE_BASIS = f"{QUANTITY_COL} × {UNIT_PRICE_COL} (verify business rules)"
else:
    raise ValueError(
        "Could not identify a revenue column or both quantity and unit price. "
        "Set the column mappings manually before continuing."
    )

print("Revenue basis:", REVENUE_BASIS)
print("Unparseable dates:", int(df[DATE_COL].isna().sum()))
print("Missing country:", int(df[COUNTRY_COL].isna().sum()))
print("Missing product:", int(df[PRODUCT_COL].isna().sum()))
print("Missing analysis revenue:", int(df["_analysis_revenue"].isna().sum()))

## 6. EDA — Missingness, Duplicates, and Numerical Summary

Interpret the measured outputs before cleaning. Do not remove duplicates or outliers automatically: repeated-looking sales records may be legitimate transactions, and negative values may represent returns or corrections.

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_report)
print(f"Exact duplicate rows: {int(df.duplicated().sum()):,}")

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    display(df[numeric_cols].describe().T)

# Revenue-specific checks
revenue_series = df["_analysis_revenue"].dropna()
print("Revenue summary:")
display(revenue_series.describe().to_frame("value"))
print("Negative revenue rows:", int((revenue_series < 0).sum()))
print("Zero revenue rows:", int((revenue_series == 0).sum()))

## 7. EDA — Transaction Date Coverage and Monthly Revenue

In [ ]:
valid_dates = df.dropna(subset=[DATE_COL]).copy()
valid_dates["_month"] = valid_dates[DATE_COL].dt.to_period("M").astype(str)

monthly_summary = (
    valid_dates.dropna(subset=["_analysis_revenue"])
    .groupby("_month", as_index=False)
    .agg(
        total_revenue=("_analysis_revenue", "sum"),
        transaction_rows=("_analysis_revenue", "size"),
        unique_products=(PRODUCT_COL, "nunique"),
        countries=(COUNTRY_COL, "nunique")
    )
    .sort_values("_month")
)

print("Observed date range:",
      valid_dates[DATE_COL].min(), "to", valid_dates[DATE_COL].max())
display(monthly_summary)

plt.figure(figsize=(11, 5))
plt.plot(monthly_summary["_month"], monthly_summary["total_revenue"], marker="o")
plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (dataset currency)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Interpretation to record after running:** Confirm that dates fall within 2019, identify missing or out-of-range dates, and investigate unusually high or low monthly totals against source records. Monthly variation may affect split design and evaluation coverage; one year of data is not enough to establish a long-term seasonal pattern.

## 8. EDA — Revenue by Country

In [ ]:
country_summary = (
    df.dropna(subset=[COUNTRY_COL, "_analysis_revenue"])
    .groupby(COUNTRY_COL, as_index=False)
    .agg(
        total_revenue=("_analysis_revenue", "sum"),
        transaction_rows=("_analysis_revenue", "size"),
        unique_products=(PRODUCT_COL, "nunique")
    )
    .sort_values("total_revenue", ascending=False)
)

display(country_summary.head(20))

top_countries = country_summary.head(10).sort_values("total_revenue")
plt.figure(figsize=(10, 6))
plt.barh(top_countries[COUNTRY_COL].astype(str), top_countries["total_revenue"])
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue (dataset currency)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

**Interpretation to record after running:** Check for inconsistent country labels and countries with very few transactions. High revenue does not necessarily mean high profitability. The country summary is also a reference output for validating the question “What was the total revenue in Australia?”

## 9. EDA — Top Products by Revenue

In [ ]:
product_summary = (
    df.dropna(subset=[PRODUCT_COL, "_analysis_revenue"])
    .groupby(PRODUCT_COL, as_index=False)
    .agg(
        total_revenue=("_analysis_revenue", "sum"),
        transaction_rows=("_analysis_revenue", "size")
    )
    .sort_values("total_revenue", ascending=False)
)

if PRODUCT_DESC_COL is not None:
    descriptions = (
        df.dropna(subset=[PRODUCT_COL, PRODUCT_DESC_COL])
        .groupby(PRODUCT_COL)[PRODUCT_DESC_COL]
        .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0])
        .rename("product_description")
    )
    product_summary = product_summary.merge(
        descriptions, on=PRODUCT_COL, how="left"
    )

display(product_summary.head(10))

top_products = product_summary.head(10).sort_values("total_revenue")
plt.figure(figsize=(10, 6))
plt.barh(top_products[PRODUCT_COL].astype(str), top_products["total_revenue"])
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue (dataset currency)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

**Interpretation to record after running:** Check whether product identifiers and descriptions are consistent. Investigate products with extreme totals and confirm how returns, cancellations, and negative transaction values are handled before treating the ranking as final.

## 10. Create a 70% Train / 15% Development / 15% Test Split

Milestone 1 proposes stratification across month, country, and product characteristics. Because product identifiers may have very high cardinality, this notebook uses a manageable composite stratum from month, country, and an optional product-category field if one exists. Rare combinations are pooled to avoid impossible strata. The distributions are then measured in all partitions. This is approximate stratification, not a guarantee that every product has identical proportions.

In [ ]:
# Optional: set this to a real, low-cardinality product category field if available.
PRODUCT_CATEGORY_COL = find_column(df.columns, [
    "product_category", "category", "product_group", "item_category"
])
print("Optional product category column:", PRODUCT_CATEGORY_COL)

split_df = df.copy()

# Date-derived month is used for stratification; invalid dates receive an explicit label.
split_df["_split_month"] = split_df[DATE_COL].dt.month.astype("Int64").astype(str)
split_df["_split_month"] = split_df["_split_month"].replace("<NA>", "unknown")

split_df["_split_country"] = (
    split_df[COUNTRY_COL].fillna("unknown").astype(str).str.strip()
)

strata_columns = ["_split_month", "_split_country"]

if PRODUCT_CATEGORY_COL is not None:
    split_df["_split_product_category"] = (
        split_df[PRODUCT_CATEGORY_COL].fillna("unknown").astype(str).str.strip()
    )
    strata_columns.append("_split_product_category")

raw_strata = split_df[strata_columns].astype(str).agg(" | ".join, axis=1)
strata_counts = raw_strata.value_counts()

# For a two-stage stratified split, each stratum must have at least 2 members
# in the first split and enough members in the temporary set. Pool rare combinations.
MIN_STRATUM_COUNT = 10
split_df["_stratum"] = raw_strata.where(
    raw_strata.map(strata_counts) >= MIN_STRATUM_COUNT,
    "__RARE_COMBINATION__"
)

print("Number of composite strata before pooling:", raw_strata.nunique())
print("Number of strata after pooling:", split_df["_stratum"].nunique())
print("Rows in pooled rare combinations:",
      int((split_df["_stratum"] == "__RARE_COMBINATION__").sum()))

# First split: 70% train, 30% temporary.
train_df, temp_df = train_test_split(
    split_df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=split_df["_stratum"]
)

# Split temporary data evenly: 15% development and 15% test.
dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["_stratum"]
)

# Remove helper columns from exported datasets; retain original data fields.
helper_cols = [
    "_split_month", "_split_country", "_split_product_category", "_stratum"
]
train_df = train_df.drop(columns=[c for c in helper_cols if c in train_df.columns])
dev_df = dev_df.drop(columns=[c for c in helper_cols if c in dev_df.columns])
test_df = test_df.drop(columns=[c for c in helper_cols if c in test_df.columns])

print(f"Train: {len(train_df):,} rows ({len(train_df)/len(df):.1%})")
print(f"Dev:   {len(dev_df):,} rows ({len(dev_df)/len(df):.1%})")
print(f"Test:  {len(test_df):,} rows ({len(test_df)/len(df):.1%})")

## 11. Validate Split Coverage and Distributions

In [ ]:
def split_profile(name, data):
    dates = pd.to_datetime(data[DATE_COL], errors="coerce")
    print(f"--- {name} ---")
    print("Rows:", f"{len(data):,}")
    print("Date range:", dates.min(), "to", dates.max())
    print("Missing date %:", round(dates.isna().mean() * 100, 2))
    print("Country distribution (top 10 proportions):")
    display(data[COUNTRY_COL].fillna("unknown").value_counts(normalize=True).head(10).rename("proportion").to_frame())
    print("Month distribution:")
    display(dates.dt.month.value_counts(normalize=True).sort_index().rename("proportion").to_frame())
    print()

split_profile("Train", train_df)
split_profile("Development", dev_df)
split_profile("Test", test_df)

# Confirm no row overlap using stable row hashes of original column values.
def row_hashes(frame):
    return pd.util.hash_pandas_object(frame, index=False)

train_hashes = set(row_hashes(train_df).tolist())
dev_hashes = set(row_hashes(dev_df).tolist())
test_hashes = set(row_hashes(test_df).tolist())

print("Train/dev hash overlap:", len(train_hashes & dev_hashes))
print("Train/test hash overlap:", len(train_hashes & test_hashes))
print("Dev/test hash overlap:", len(dev_hashes & test_hashes))

**Interpretation:** Verify the proportions, month coverage, and country distributions. If a country or month is very rare, exact matching across all splits may not be possible. Duplicate rows require special care: identical records can occur legitimately in transaction data, so row-hash overlap is a diagnostic rather than proof of leakage. This split is intended for the project’s stated analytical use and is not a future-sales forecasting split.

## 12. Save Versioned Parquet, Splits, and JSON Metadata

In [ ]:
created = datetime.now(timezone.utc)
dataset_version = created.strftime("%Y%m%dT%H%M%SZ")
processing_version = "v1"

output_root = Path(LOCAL_OUTPUT_DIR)
processed_dir = output_root / "processed"
splits_dir = output_root / "splits"
processed_dir.mkdir(parents=True, exist_ok=True)
splits_dir.mkdir(parents=True, exist_ok=True)

# Export processed data with the derived analysis revenue field included.
processed_df = df.copy()
processed_df.to_parquet(
    processed_dir / f"sales_processed_{dataset_version}.parquet",
    index=False
)

for name, partition in [
    ("train", train_df), ("dev", dev_df), ("test", test_df)
]:
    partition.to_parquet(
        splits_dir / f"{name}_{dataset_version}.parquet",
        index=False
    )

metadata = {
    "dataset_version": dataset_version,
    "created": created.isoformat(),
    "source": f"gs://{BUCKET_NAME}/{RAW_GCS_PATH}",
    "source_sha256": source_sha256,
    "rows": int(len(df)),
    "processing_version": processing_version,
    "split_seed": RANDOM_STATE,
    "split_proportions": {
        "train": len(train_df) / len(df),
        "dev": len(dev_df) / len(df),
        "test": len(test_df) / len(df)
    },
    "date_column": str(DATE_COL),
    "country_column": str(COUNTRY_COL),
    "product_column": str(PRODUCT_COL),
    "revenue_basis": REVENUE_BASIS
}

metadata_path = output_root / f"metadata_{dataset_version}.json"
metadata_path.write_text(json.dumps(metadata, indent=2))

print("Dataset version:", dataset_version)
print("Metadata:", metadata_path)
print("Output files:")
for path in sorted(output_root.rglob("*")):
    if path.is_file():
        print(f"{path} ({path.stat().st_size:,} bytes)")

## 13. Upload Processed Data and Splits to GCS

In [ ]:
def upload_folder_to_gcs(local_folder, bucket_name, gcs_prefix):
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)
    uploaded = []

    for root, _, files in os.walk(local_folder):
        for file_name in sorted(files):
            local_path = os.path.join(root, file_name)
            relative_path = os.path.relpath(local_path, local_folder).replace(os.sep, "/")
            object_path = f"{gcs_prefix.rstrip('/')}/{relative_path}"
            bucket.blob(object_path).upload_from_filename(local_path)
            uploaded.append(object_path)
            print(f"Uploaded: gs://{bucket_name}/{object_path}")

    return uploaded

uploaded_objects = upload_folder_to_gcs(
    str(output_root),
    BUCKET_NAME,
    f"{PROCESSED_PREFIX}/{dataset_version}"
)

print(f"Uploaded {len(uploaded_objects)} files.")

## 14. Optional: Load Processed Data into BigQuery

Run this section only after confirming the target dataset exists and the service account has permission to create or replace the intended table. This example creates a versioned table name to avoid silently overwriting a previous dataset version.

In [ ]:
# The processed DataFrame is exported to a local Parquet file above.
processed_parquet_path = processed_dir / f"sales_processed_{dataset_version}.parquet"

BQ_TABLE_ID = (
    f"{PROJECT_ID}.{BQ_DATASET}."
    f"{BQ_TABLE_PREFIX}_{dataset_version.lower()}"
)

# Optional setup: create the dataset if it does not exist.
bq_client = bigquery.Client(project=PROJECT_ID)
dataset_id = f"{PROJECT_ID}.{BQ_DATASET}"

try:
    bq_client.get_dataset(dataset_id)
    print("BigQuery dataset exists:", dataset_id)
except Exception as exc:
    print(
        "BigQuery dataset was not found or is not accessible. "
        "Create it or confirm permissions before loading."
    )
    raise exc

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_EMPTY
)

with open(processed_parquet_path, "rb") as parquet_file:
    load_job = bq_client.load_table_from_file(
        parquet_file,
        BQ_TABLE_ID,
        job_config=job_config
    )

load_job.result()
bq_table = bq_client.get_table(BQ_TABLE_ID)

print("Loaded table:", BQ_TABLE_ID)
print("BigQuery row count:", bq_table.num_rows)
print("BigQuery schema:", [(field.name, field.field_type) for field in bq_table.schema])

## 15. Verify Selected Aggregates

Compare Python and BigQuery results for the same dataset version. The query below uses the derived `_analysis_revenue` column, and assumes the table loaded in the previous section.

In [ ]:
# Python reference aggregate
python_country_total = (
    df.loc[
        df[COUNTRY_COL].astype(str).str.strip().str.casefold().eq("australia"),
        "_analysis_revenue"
    ].sum()
)

print("Python Australia revenue:", python_country_total)

# BigQuery reference aggregate; quote column names safely with backticks.
# Column identifiers below come from the configured schema, not user question text.
country_col_sql = f"`{COUNTRY_COL}`"
revenue_col_sql = "`_analysis_revenue`"

query = f'''
SELECT SUM({revenue_col_sql}) AS australia_revenue
FROM `{BQ_TABLE_ID}`
WHERE LOWER(TRIM(CAST({country_col_sql} AS STRING))) = "australia"
'''

query_job = bq_client.query(query)
bq_result = list(query_job.result())[0]["australia_revenue"]

print("BigQuery Australia revenue:", bq_result)

if bq_result is not None and pd.notna(python_country_total):
    print("Absolute difference:", abs(float(bq_result) - float(python_country_total)))
else:
    print("Comparison unavailable: no matching values or null aggregate.")

**Validation note:** Do not treat the comparison as passed until the two results agree within the chosen tolerance. If they differ, check type conversion, null handling, revenue definition, source version, and whether the BigQuery table contains the same records as the local DataFrame.

## 16. Verify GCS Objects and Read Back a Parquet File

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
prefix = f"{PROCESSED_PREFIX}/{dataset_version}/"
blobs = list(storage_client.list_blobs(BUCKET_NAME, prefix=prefix))

print(f"Objects under gs://{BUCKET_NAME}/{prefix}")
for blob in blobs:
    print(f"{blob.name} ({blob.size:,} bytes)")

required_suffixes = [
    f"processed/sales_processed_{dataset_version}.parquet",
    f"splits/train_{dataset_version}.parquet",
    f"splits/dev_{dataset_version}.parquet",
    f"splits/test_{dataset_version}.parquet",
    f"metadata_{dataset_version}.json"
]
found_names = {blob.name for blob in blobs}
missing = [
    f"{prefix}{suffix}" for suffix in required_suffixes
    if f"{prefix}{suffix}" not in found_names
]

if missing:
    raise ValueError(f"Missing expected GCS objects: {missing}")

print("All expected versioned outputs are present.")

# Smoke test: read the training split back from GCS.
train_blob_path = (
    f"gs://{BUCKET_NAME}/{prefix}"
    f"splits/train_{dataset_version}.parquet"
)
train_readback = pd.read_parquet(train_blob_path)
print("Read back training rows:", len(train_readback))
print("Read back columns:", train_readback.columns.tolist())
display(train_readback.head(3))

## 17. Final Summary and Evidence Checklist

In [ ]:
summary = {
    "raw_rows_loaded": int(len(df)),
    "reported_milestone_rows": EXPECTED_ROWS_FROM_MILESTONE_1,
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "unparseable_dates": int(df[DATE_COL].isna().sum()),
    "missing_country": int(df[COUNTRY_COL].isna().sum()),
    "missing_product": int(df[PRODUCT_COL].isna().sum()),
    "missing_analysis_revenue": int(df["_analysis_revenue"].isna().sum()),
    "train_rows": int(len(train_df)),
    "dev_rows": int(len(dev_df)),
    "test_rows": int(len(test_df)),
    "dataset_version": dataset_version,
    "processing_version": processing_version,
    "revenue_basis": REVENUE_BASIS
}
print(json.dumps(summary, indent=2))

### Before submitting Milestone 2

- [ ] Record the real dataset source URL/DOI, owner, licence, and permissions.
- [ ] Capture the actual schema, missingness, duplicate count, and data-type checks.
- [ ] Include the monthly revenue chart and country/product summaries with interpretations.
- [ ] Investigate anomalous or negative revenue values rather than automatically deleting them.
- [ ] Confirm the train/dev/test proportions and distributions.
- [ ] Verify the versioned Parquet files and JSON metadata in GCS.
- [ ] If BigQuery is enabled, capture the loaded table row count and compare reference aggregates.
- [ ] Report only results produced by an actual notebook execution.

**Limitations:** This notebook is a template based on the Milestone 1 PDF. The PDF does not specify the raw CSV schema, original source URL/licence, GCP project, bucket, or object path. Cloud cells require configuration and access to the real dataset. No data-derived statistics are pre-filled or fabricated.